# Manipulating Statements

First import the necessary classes and `pandas`.

In [ ]:
from pathlib import Path

import pandas as pd

from finstmt import FinancialStatements
from finstmt.config.statement import STATEMENT_CONFIGS

## Load Financial Statements

In [ ]:
data_folder = Path("..") / "tests" / "sources" / "stockrow" / "CAT"
inc_df = pd.read_csv(data_folder / "annual_income.csv", index_col=0)
bs_df = pd.read_csv(data_folder / "annual_bs.csv", index_col=0)

stmts = FinancialStatements.from_df(
    pd.concat([inc_df, bs_df]), STATEMENT_CONFIGS, disp_unextracted=False
)

## View Statements

In [ ]:
stmts

## Select Statement Periods

Select periods by indexing the statements with them:

In [ ]:
stmts["12-31-2018"]  # one period

Now select multiple periods. The date format doesn't matter:

In [ ]:
stmts[["12/31/2015", "12-31-2018"]]

## Statement Math

All the math operators work on statements. With a number, the operation applies to every item. With other statements, items in overlapping periods are combined pairwise, and non-overlapping periods are put together.

### Adjust Units

In [ ]:
stmts / 1_000_000  # in millions

### Math with Overlapping Statements

When the periods overlap, the math applies to each pair of items. This helps when aggregating subsidiaries or segments:

In [ ]:
stmts + stmts

### Combine Non-Overlapping Statements

Adding statements with different periods puts all the periods in one set of statements:

In [ ]:
stmts["12/31/2018"] + stmts["12/31/2014"]

This is useful after a forecast, when a calculation such as free cash flow needs the last historical period:

In [ ]:
for key in ["cash_and_st_invest", "net_ppe", "op_exp"]:
    stmts.config.update(key, ["forecast", "make_forecast"], False)

stmts.config.update_all(["forecast", "method"], "trend")
fcst = stmts.forecast(periods=2)
(stmts[stmts.dates[-1]] + fcst).fcf.dropna()